# GPU tier smoke testRun this on Kaggle with a **GPU accelerator and Internet enabled** beforecommitting GPU hours to the twelve outstanding models.The point is to find out what breaks. Nothing here is a benchmark run: oneticker, ten sessions, one epoch. Every cell is independently runnable — eachcarries its own imports, its own data, and its own installs — so a cell thatfails can be re-run alone after a fix without replaying the notebook.Each cell ends in a single `PASS` or `FAIL` line. Cell 3 is expected to be thenoisy one; it reports and does not debug.**Ordering is deliberate.** Kronos runs first because it is the riskiest: aseparate repository rather than a pip package, a separate API, its owntokeniser, and OHLCV input where everything else takes a univariate series. Ifit fails, that is worth knowing before any budget is spent.Reference: `configs/models.yaml` and `src/sp500bench/models/registry.py` are thesource of truth for checkpoints. The five foundation entries are Chronos-2,TimesFM-2.5, Moirai-2.0, Kronos-small and TTM.

## What the first run established (2 September 2026)Kaggle, Tesla T4 15.6 GB, torch 2.10, CUDA 12.8.| model | outcome ||---|---|| **Kronos-small** | **PASS** — loads and runs || **Moirai-2.0** | **LOADED** — predictor constructs || Chronos-2 | FAIL — `ImportError: cannot import name '_torch_version' from transformers.utils.import_utils` || TTM | FAIL — `ImportError: cannot import name 'is_tf_available' from transformers.utils` || TimesFM-2.5 | FAIL — `AttributeError: module 'timesfm' has no attribute 'TimesFm'` || DLinear (cell 4) | FAIL — `RuntimeError: Lightning can't create new processes if CUDA is already initialized` |**The two ImportErrors are one failure, not two.** Kaggle ships a transformersrelease that removed both `_torch_version` and `is_tf_available`; Chronos-2 andTTM each import one of them. Neither is a problem with the checkpoint. Cell 3now pins transformers to the 4.x line *after* installing the model packages, sonothing upgrades it back, and prints the version pip actually resolved.**TimesFM was an API guess on my part and it was wrong.** `timesfm.TimesFm` doesnot exist in the installed release. Cell 3 no longer guesses: it prints`dir(timesfm)` and the version, then constructs from whatever entry point isactually present. If none is recognisable it reports the namespace and stops —that is a useful outcome, not a failure to handle.**Cell 4's failure is an ordering artefact, not a DLinear problem.** Lightningcannot fork worker processes once CUDA has been initialised, and cells 2 and 3initialise it. Fixed here with `num_workers_loader=0` and an explicitsingle-device GPU trainer, but see the warning on cell 4: **the real neural-tierrun belongs in its own Kaggle session**, not after the foundation tier.**Two things Kronos passing has already settled**, both of which the evaluationlayer has to honour:- it emits **price-level OHLCV**, so returns must be derived from the predicted  close rather than read off directly;- it samples autoregressively, so its **CRPS must come from sampled paths**, not  from a closed-form quantile head. `sample_count=5` here is a smoke test; a  real run needs far more, and that is a compute cost the budget has to carry.Both are already recorded in `registry.py` and `configs/models.yaml`.

In [ ]:
# ============================================================# CELL 1 - ENVIRONMENT# ============================================================!pip install -q neuralforecast statsforecast pyyamlimport sysprint("=" * 60)ok = Truetry:    import torch    print(f"torch              {torch.__version__}")    cuda = torch.cuda.is_available()    print(f"CUDA available     {cuda}")    if cuda:        print(f"GPU                {torch.cuda.get_device_name(0)}")        print(f"CUDA version       {torch.version.cuda}")        total = torch.cuda.get_device_properties(0).total_memory / 1e9        print(f"GPU memory         {total:.1f} GB")    else:        print("GPU                none - accelerator is off or unavailable")        ok = Falseexcept Exception as exc:    print(f"torch import failed: {type(exc).__name__}: {exc}")    ok = Falsefor name in ("neuralforecast", "statsforecast", "yaml"):    try:        module = __import__(name)        print(f"{name:<19}{getattr(module, '__version__', 'ok')}")    except Exception as exc:        print(f"{name:<19}FAILED: {type(exc).__name__}: {exc}")        ok = Falseprint(f"python             {sys.version.split()[0]}")print("=" * 60)print("PASS - environment ready" if ok else      "FAIL - no GPU or a core package is missing; fix before running cells 2-4")

In [ ]:
# ============================================================# CELL 2 - KRONOS  (riskiest first: separate repo, separate API)# ============================================================import sysimport warningsimport numpy as npimport pandas as pdwarnings.filterwarnings("ignore")TICKER = "AAPL"CONTEXT = 400          # Kronos-small documents a 512-session context limitPRED_LEN = 10SAMPLE_COUNT = 5# Idempotent: the guard is shell-side so the cell can be re-run alone# without git erroring on an existing directory.![ -d Kronos ] || git clone --depth 1 https://github.com/shiyu-coder/Kronos.gitif "Kronos" not in sys.path:    sys.path.append("Kronos")def load_bars(ticker, n):    """OHLCV bars. Kronos is the one model here that takes them natively."""    try:        import yfinance as yf        frame = yf.download(ticker, period="3y", progress=False, auto_adjust=True)        if len(frame):            if isinstance(frame.columns, pd.MultiIndex):                frame.columns = frame.columns.get_level_values(0)            frame = frame.rename(columns=str.lower)[                ["open", "high", "low", "close", "volume"]].dropna()            return frame.tail(n), "yfinance"    except Exception as exc:        print(f"  yfinance unavailable ({type(exc).__name__}); using synthetic bars")    rng = np.random.default_rng(20260902)    close = 100 * np.exp(np.cumsum(rng.normal(0, 0.015, n)))    frame = pd.DataFrame({        "open": close * (1 + rng.normal(0, 0.002, n)),        "high": close * (1 + abs(rng.normal(0, 0.006, n))),        "low": close * (1 - abs(rng.normal(0, 0.006, n))),        "close": close,        "volume": rng.integers(1e6, 5e6, n).astype(float),    }, index=pd.bdate_range("2023-01-02", periods=n))    return frame, "synthetic"print("=" * 60)try:    from model import Kronos, KronosPredictor, KronosTokenizer    tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")    model = Kronos.from_pretrained("NeoQuasar/Kronos-small")    print("checkpoint + tokenizer loaded")    import torch    device = "cuda:0" if torch.cuda.is_available() else "cpu"    predictor = KronosPredictor(model, tokenizer, device=device, max_context=512)    bars, source = load_bars(TICKER, CONTEXT + PRED_LEN)    history, future = bars.iloc[:-PRED_LEN], bars.iloc[-PRED_LEN:]    print(f"data               {source}, {len(history)} context sessions")    out = predictor.predict(        df=history[["open", "high", "low", "close", "volume"]],        x_timestamp=pd.Series(history.index),        y_timestamp=pd.Series(future.index),        pred_len=PRED_LEN,        T=1.0,        top_p=0.9,        sample_count=SAMPLE_COUNT,        verbose=False,    )    print(f"device             {device}")    print(f"output type        {type(out).__name__}")    print(f"output shape       {getattr(out, 'shape', 'n/a')}")    print(f"output columns     {list(getattr(out, 'columns', []))}")    print(out.head(3) if hasattr(out, "head") else out)    print()    print("NOTE: CRPS for Kronos must come from sampled paths, not a quantile")    print(f"head. sample_count={SAMPLE_COUNT} here; a real run needs far more.")    print("=" * 60)    print(f"PASS - Kronos ran, {PRED_LEN} sessions x {SAMPLE_COUNT} samples")except Exception as exc:    import traceback    traceback.print_exc()    print("=" * 60)    print(f"FAIL - Kronos: {type(exc).__name__}: {exc}")

In [ ]:
# ============================================================# CELL 3 - THE OTHER FOUR FOUNDATION CHECKPOINTS# One load + one call each. Report only - do not debug here.# ============================================================# Order matters. The model packages are installed first, then transformers is# pinned back to the 4.x line, so nothing pulls a 5.x release in behind us.# Kaggle's default transformers removed both `_torch_version` (which# chronos-forecasting imports) and `is_tf_available` (which tsfm_public# imports); that single incompatibility is what failed Chronos-2 and TTM, not# anything about the checkpoints. No patch version is guessed here - a <5# constraint goes in and whatever pip resolves is printed below.!pip install -q chronos-forecasting timesfm uni2ts granite-tsfm 2>/dev/null!pip install -q "transformers<5" 2>/dev/nullimport sysimport warningsimport numpy as npimport pandas as pdwarnings.filterwarnings("ignore")CONTEXT = 512PRED_LEN = 10# Report what is on disk *and* what is loaded in this kernel. pip cannot change# a module that a previous cell already imported, so if these disagree the cell# needs a kernel restart before its result means anything.from importlib.metadata import version as dist_versiontry:    on_disk = dist_version("transformers")except Exception as exc:    on_disk = f"unknown ({type(exc).__name__})"already_imported = "transformers" in sys.modulesin_memory = getattr(sys.modules.get("transformers"), "__version__", None)print(f"transformers on disk       {on_disk}")if already_imported:    print(f"transformers in kernel     {in_memory}")    if str(in_memory) != str(on_disk):        print("  ** RESTART THE KERNEL AND RE-RUN THIS CELL **")        print("  A previous cell imported transformers, so the pin above has")        print("  not taken effect in this session and the results below are")        print("  from the old version.")else:    print("transformers in kernel     not yet imported - pin is live")print()def load_series(n):    """One univariate close series; these four take a series, not OHLCV."""    try:        import yfinance as yf        frame = yf.download("AAPL", period="3y", progress=False, auto_adjust=True)        if len(frame):            if isinstance(frame.columns, pd.MultiIndex):                frame.columns = frame.columns.get_level_values(0)            column = "Close" if "Close" in frame else "close"            return frame[column].dropna().tail(n).to_numpy(), "yfinance"    except Exception:        pass    rng = np.random.default_rng(20260902)    return 100 * np.exp(np.cumsum(rng.normal(0, 0.015, n))), "synthetic"values, source = load_series(CONTEXT)print(f"context: {len(values)} sessions from {source}\n")results = {}def attempt(name, checkpoint, fn):    try:        outcome = fn()        results[name] = ("LOADED", str(outcome))        print(f"  {name:<14} LOADED   {checkpoint}  -> {outcome}")    except Exception as exc:        detail = f"{type(exc).__name__}: {exc}"        results[name] = ("FAILED", detail)        print(f"  {name:<14} FAILED   {checkpoint}")        print(f"                 {detail[:200]}")def run_chronos():    import torch    from chronos import BaseChronosPipeline    pipe = BaseChronosPipeline.from_pretrained(        "amazon/chronos-2",        device_map="cuda" if torch.cuda.is_available() else "cpu")    out = pipe.predict(torch.tensor(values, dtype=torch.float32), PRED_LEN)    return tuple(np.asarray(out).shape)def run_timesfm():    """Do not guess the API. Print the namespace, then use what is there.    `timesfm.TimesFm` was a guess and it does not exist in the installed    release. This inspects the module and tries the entry points it actually    exposes; if none is recognisable it reports the namespace and raises, which    is a useful smoke-test outcome rather than a failure to handle.    """    import timesfm    print(f"    timesfm version  {getattr(timesfm, '__version__', 'unknown')}")    public = sorted(n for n in dir(timesfm) if not n.startswith("_"))    print(f"    dir(timesfm)     {public}")    candidates = [n for n in public                  if "timesfm" in n.lower() and n[:1].isupper()                  and isinstance(getattr(timesfm, n), type)]    print(f"    candidates       {candidates or 'none'}")    if not candidates:        raise AttributeError(            "no TimesFM entry point found; namespace printed above. "            "Read it and pick the constructor before spending GPU hours.")    errors = []    for name in candidates:        cls = getattr(timesfm, name)        try:            model = cls()                       # 2.5-style: construct, then load            for loader in ("load_checkpoint", "from_pretrained"):                if hasattr(model, loader):                    getattr(model, loader)()                    break            return f"constructed via timesfm.{name}"        except Exception as exc:            errors.append(f"{name}: {type(exc).__name__}: {str(exc)[:80]}")    raise RuntimeError("no candidate constructed -> " + " | ".join(errors))def run_moirai():    from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module    module = Moirai2Module.from_pretrained("Salesforce/moirai-2.0-R-small")    forecast = Moirai2Forecast(module=module, prediction_length=PRED_LEN,                               context_length=len(values),                               target_dim=1, feat_dynamic_real_dim=0,                               past_feat_dynamic_real_dim=0)    predictor = forecast.create_predictor(batch_size=1)    return f"constructed {type(predictor).__name__}"def run_ttm():    import torch    from tsfm_public.models.tinytimemixer import TinyTimeMixerForPrediction    model = TinyTimeMixerForPrediction.from_pretrained(        "ibm-granite/granite-timeseries-ttm-r2")    ctx = model.config.context_length    x = torch.tensor(values[-ctx:], dtype=torch.float32).reshape(1, ctx, 1)    with torch.no_grad():        out = model(past_values=x)    return tuple(out.prediction_outputs.shape)print("=" * 60)attempt("Chronos-2", "amazon/chronos-2", run_chronos)attempt("TimesFM-2.5", "google/timesfm-2.5-200m-pytorch", run_timesfm)attempt("Moirai-2.0", "Salesforce/moirai-2.0-R-small", run_moirai)attempt("TTM", "ibm-granite/granite-timeseries-ttm-r2", run_ttm)print("=" * 60)loaded = [k for k, (s, _) in results.items() if s == "LOADED"]failed = [k for k, (s, _) in results.items() if s == "FAILED"]print(f"transformers resolved to {on_disk}")print(f"loaded ({len(loaded)}): {', '.join(loaded) if loaded else 'none'}")print(f"failed ({len(failed)}): {', '.join(failed) if failed else 'none'}")print("PASS - all four loaded" if not failed else      f"FAIL - {len(failed)} of 4 did not load: {', '.join(failed)}")print()print("Reporting only. Do not debug here - record what broke, then decide")print("whether the integration cost is worth the GPU budget.")

In [ ]:
# ============================================================# CELL 4 - NEURALFORECAST SMOKE TEST (DLinear, MQLoss, 9-point grid)# ============================================================# RUN THE REAL NEURAL TIER IN ITS OWN KAGGLE SESSION.## Lightning forks worker processes, and it cannot fork once CUDA has been# initialised in the interpreter. Cells 2 and 3 both initialise it, so running# this cell after them raised:##     RuntimeError: Lightning can't create new processes if CUDA is already#     initialized## num_workers_loader=0 keeps Lightning from spawning at all, which makes the# smoke test survive that ordering. It is a workaround, not a fix: it also# serialises data loading, which is fine for one epoch on five tickers and# wrong for seven models over 50 tickers and four folds. The foundation tier and# the neural tier belong in separate sessions - a fresh kernel per tier, so the# neural run gets real dataloader workers and an uninitialised CUDA context.# ============================================================import osimport warningsimport numpy as npimport pandas as pdwarnings.filterwarnings("ignore")TICKERS = ["AAPL", "MSFT", "JPM", "XOM", "JNJ"]HORIZON = 1LOOKBACK = 60# The grid must match configs/eval.yaml exactly - training.quantiles and the# evaluation grid are checked for equality at load time by# sp500bench.config.load_configs. If the repo is not on this machine the literal# is used, and the source is printed so a silent divergence is visible.GRID = [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]grid_source = "literal fallback (repo not found)"for candidate in ("configs/eval.yaml", "../configs/eval.yaml",                  "/kaggle/working/sp500-forecast-benchmark/configs/eval.yaml"):    if os.path.exists(candidate):        import yaml        GRID = yaml.safe_load(open(candidate))["quantiles"]        grid_source = candidate        breakprint(f"quantile grid ({len(GRID)}): {GRID}")print(f"source: {grid_source}\n")def load_panel(tickers):    try:        import yfinance as yf        frame = yf.download(tickers, period="3y", progress=False, auto_adjust=True)        close = frame["Close"] if "Close" in frame else frame["close"]        long = (close.stack().rename("price").reset_index())        long.columns = ["ds", "unique_id", "price"]        long["y"] = long.groupby("unique_id")["price"].transform(            lambda s: np.log(s).diff())        return long.dropna()[["unique_id", "ds", "y"]], "yfinance"    except Exception:        rng = np.random.default_rng(20260902)        dates = pd.bdate_range("2023-01-02", periods=600)        rows = [pd.DataFrame({"unique_id": t, "ds": dates,                              "y": rng.normal(0, 0.018, len(dates))})                for t in tickers]        return pd.concat(rows, ignore_index=True), "synthetic"print("=" * 60)try:    from neuralforecast import NeuralForecast    from neuralforecast.losses.pytorch import MQLoss    from neuralforecast.models import DLinear    panel, source = load_panel(TICKERS)    print(f"panel              {source}, {len(panel):,} rows, "          f"{panel.unique_id.nunique()} tickers")    import torch    on_gpu = torch.cuda.is_available()    model = DLinear(        h=HORIZON, input_size=LOOKBACK, loss=MQLoss(quantiles=GRID),        max_steps=1, val_check_steps=1, enable_progress_bar=False,        logger=False, scaler_type="standard",        # No forked dataloader workers: see the warning at the top of the cell.        num_workers_loader=0,        # Pin to one device explicitly rather than letting Lightning choose a        # strategy; "auto" can select a spawning one on a multi-GPU box.        accelerator="gpu" if on_gpu else "cpu",        devices=1,    )    print(f"trainer            accelerator={'gpu' if on_gpu else 'cpu'}, "          f"devices=1, num_workers_loader=0")    nf = NeuralForecast(models=[model], freq="B")    nf.fit(df=panel, verbose=False)    forecast = nf.predict()    columns = list(forecast.columns)    print(f"forecast shape     {forecast.shape}")    print(f"forecast columns   {columns}")    # Do not assume the naming scheme - count what actually came back. Different    # neuralforecast versions spell MQLoss outputs differently (-q-1.0, -lo-98,    # -median), and the evaluation layer has to read whatever this is.    output_columns = [c for c in columns if c not in ("unique_id", "ds")]    print(f"\n{len(output_columns)} output column(s) for {len(GRID)} quantiles:")    for c in output_columns:        print(f"  {c}")    emitted = len(output_columns)    print("=" * 60)    if emitted == len(GRID):        print(f"PASS - DLinear emitted all {len(GRID)} quantile columns")        print("       Reminder: run the real neural tier in a fresh session.")    else:        print(f"FAIL - expected {len(GRID)} quantile columns, got {emitted}")        print("       Map these names to the q0.01..q0.99 convention in")        print("       sp500bench.config.quantile_column before any real run.")except Exception as exc:    import traceback    traceback.print_exc()    print("=" * 60)    print(f"FAIL - neuralforecast: {type(exc).__name__}: {exc}")